In [12]:
from getdist import loadMCSamples
import getdist.plots as plots
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [13]:
class CobayaAnalyzer:
    def __init__(self, chain_dicts, burn_in=0.3, best_sample=False):
        self.chains = chain_dicts.copy()
        self.burn_in = burn_in
        self.best_sample = best_sample
        self.bestfit = {}
        self._load_chains()
    def _load_chains(self):
        print(f"Loading chains")
        for experiment, [chain_path, _, _] in self.chains.items():
            try:
                self.chains[experiment][0] = loadMCSamples(chain_path, settings={'ignore_rows': self.burn_in,
                                                             'smooth_scale_1D': -1, 
                                                             'smooth_scale_2D': -1})
                self.bestfit[experiment] = self.chains[experiment][0].getParamBestFitDict(best_sample=self.best_sample)
                self.chains[experiment][0].getParamNames().parWithName('omega_b').label = r'\Omega_{\rm b} h^2'
            except Exception as e:
                print(f" -> Error loading {experiment}: {e}")
    def summary_table(self, params, limit=1, marg_stat=True, bestfit_stat=True):
        table_latex = ''
        model_table = {}
        for experiment, [sample, dataset, model] in self.chains.items():
            if model not in model_table.keys():
                model_table[model] = model + " & "*len(params) + "\\\\ \n"

            # Report each line with the result from each dataset with the same model
            model_table[model] += dataset
            best_fit_params = self.bestfit[experiment]
            for param in params:
                if marg_stat:
                    try:
                        model_table[model] += " & " + "$" + sample.getInlineLatex(param, limit=limit).replace(">","=>").replace("<","=<").split("=", 1)[1]+ "$"
                    except:
                        model_table[model] += " & --- "

                if bestfit_stat:
                    try:
                        if param == 'abs_delta_qtm':
                            model_table[model] += " & " + "$" + f'{abs(best_fit_params["delta_qtm"]):.3e}' + "$"
                        else:
                            model_table[model] += " & " + "$" + f'{best_fit_params[param]:.3f}' + "$"
                    except:
                        model_table[model] += " & --- "
            model_table[model] += " \\\\ \n" 
        for table in model_table.values():
            table_latex += table
            table_latex += "\\hline\n"

        print(table_latex)

In [26]:
# Set Main Chain Dictionary
chains = {}
models = {}
chain_dir = Path(r'/home/theppawan/cosmo-research/quintom-project/chains')
chain_prefixes = [p.with_suffix('') for p in chain_dir.rglob('*.checkpoint')]

for prefix_path in chain_prefixes:
    chain_name = prefix_path.name
    model, dataset = chain_name.split("_", 1)

    # Set model label
    if 'lcdm' in model:
        models_label = r'\Lambda {\rm CDM}'
    else:
        models_label = model.capitalize()
    models[model] = models_label

    # Set dataset label
    if 'theta' in dataset:
        label = dataset.replace("theta", r"$\theta_*$")
    else:
        label = dataset
    chains[chain_name] = [str(prefix_path), rf"{label}", models_label]

# Sub-Dictionary
lcdm_chains = {key: chains[key] for key in chains.keys() if 'lcdm' in key}
quintom_chains = {key: chains[key] for key in chains.keys() if 'quintom' in key}

In [15]:
all_results = CobayaAnalyzer(chain_dicts=chains, burn_in=0.3, best_sample=True)
for _, [sample, _, _] in all_results.chains.items():
    p = sample.getParams()
    try:
        if not sample.paramNames.parWithName('abs_delta_qtm'):
            abs_delta_qtm = np.abs(p.delta_qtm)
            sample.addDerived(abs_delta_qtm, name='abs_delta_qtm', label=r'|\delta_{\mathrm{qtm}}|')
    except AttributeError:
        print(f"Parameter 'delta_qtm' not found in sample: {sample.getName()}")
        continue
all_results.summary_table(params=['H0', 'Omega_m', 'lambda_qtm', 'abs_delta_qtm'], limit=1)

Loading chains
Parameter 'delta_qtm' not found in sample: lcdm_DESI+CMB+SNIa
Parameter 'delta_qtm' not found in sample: lcdm_CMB
Parameter 'delta_qtm' not found in sample: lcdm_DESI
Parameter 'delta_qtm' not found in sample: lcdm_DESI+BBN+theta
Parameter 'delta_qtm' not found in sample: lcdm_DESI+BBN
Parameter 'delta_qtm' not found in sample: lcdm_DESI+CMB


\Lambda {\rm CDM} &  &  &  & \\ 
DESI+CMB+SNIa & $ 67.96\pm 0.40$ & $67.959$ & $ 0.3083\pm 0.0053$ & $0.308$ & ---  & ---  & ---  & ---  \\ 
CMB & $ 66.89\pm 0.61$ & $66.834$ & $ 0.3232\pm 0.0087$ & $0.324$ & ---  & ---  & ---  & ---  \\ 
DESI & $ 69^{+8}_{-4}$ & $67.533$ & $ 0.295^{+0.013}_{-0.016}$ & $0.294$ & ---  & ---  & ---  & ---  \\ 
DESI+BBN+$\theta_*$ & $ 68.60^{+0.60}_{-0.69}$ & $68.651$ & $ 0.2936\pm 0.0075$ & $0.293$ & ---  & ---  & ---  & ---  \\ 
DESI+BBN & $ 68.70\pm 0.80$ & $68.565$ & $ 0.295\pm 0.015$ & $0.293$ & ---  & ---  & ---  & ---  \\ 
DESI+CMB & $ 68.04\pm 0.41$ & $68.043$ & $ 0.3071\pm 0.0054$ & $0.307$ & ---  & ---  & ---  & ---  \\ 
\hline
Quintom &  &  &  & \\ 
DESI+BBN+SNIa & $ 61.6^{+6.6}_{-3.0}$ & $63.999$ & $ 0.311^{+0.013}_{-0.016}$ & $0.303$ & $ 0.60^{+0.34}_{-0.21}$ & $0.792$ & $ 0.087^{+0.033}_{-0.083}$ & $6.098e-02$ \\ 
DESI+CMB+SNIa+SH0ES & $ 68.49^{+0.46}_{-0.40}$ & $68.736$ & $ 0.2983\pm 0.0049$ & $0.298$ & $< 0.274$ & $0.053$ & $ 0.0170^{+0.00

In [25]:
all_results_1 = CobayaAnalyzer(chain_dicts=chains, burn_in=0.3, best_sample=True)
for _, [sample, _, _] in all_results_1.chains.items():
    p = sample.getParams()
    try:
        if not sample.paramNames.parWithName('abs_delta_qtm'):
            abs_delta_qtm = np.abs(p.delta_qtm)
            sample.addDerived(abs_delta_qtm, name='abs_delta_qtm', label=r'|\delta_{\mathrm{qtm}}|')
    except AttributeError:
        print(f"Parameter 'delta_qtm' not found in sample: {sample.getName()}")
        continue
all_results_1.summary_table(params=['sigma8'], limit=1, bestfit_stat=False)

Loading chains
Parameter 'delta_qtm' not found in sample: lcdm_DESI+CMB+SNIa
Parameter 'delta_qtm' not found in sample: lcdm_CMB
Parameter 'delta_qtm' not found in sample: lcdm_DESI
Parameter 'delta_qtm' not found in sample: lcdm_DESI+BBN+theta
Parameter 'delta_qtm' not found in sample: lcdm_DESI+BBN
Parameter 'delta_qtm' not found in sample: lcdm_DESI+CMB


\Lambda {\rm CDM} & \\ 
DESI+CMB+SNIa & $ 0.8084\pm 0.0032$ \\ 
CMB & $ 0.8165\pm 0.0047$ \\ 
DESI & $ 0.800^{+0.051}_{-0.062}$ \\ 
DESI+BBN+$\theta_*$ & $ 0.7954^{+0.0057}_{-0.0051}$ \\ 
DESI+BBN & $ 0.800\pm 0.037$ \\ 
DESI+CMB & $ 0.8077\pm 0.0033$ \\ 
\hline
Quintom & \\ 
DESI+BBN+SNIa & $ 0.607^{+0.21}_{-0.094}$ \\ 
DESI+CMB+SNIa+SH0ES & $ 0.791^{+0.013}_{-0.0042}$ \\ 
DESI+SNIa+SH0ES & $ 0.813\pm 0.045$ \\ 
DESI+CMB+SNIa & $ 0.742\pm 0.028$ \\ 
DESI & $ 0.623^{+0.19}_{-0.091}$ \\ 
DESI+BBN+SNIa+SH0ES & $ 0.863\pm 0.033$ \\ 
DESI+BBN & $ 0.606^{+0.20}_{-0.089}$ \\ 
DESI+CMB & $ 0.739\pm 0.030$ \\ 
\hline

